# Python for Data Science — Patterns & Pitfalls
> **Level:** Intermediate–Advanced | Runnable examples for every concept

## Table of Contents
1. [Missing Data Handling](#missing)
2. [Overfitting — Causes & Solutions](#overfitting)
3. [High Cardinality Encoding](#encoding)
4. [NumPy — Reproducibility & Arrays](#numpy)
5. [Outliers — When to Keep, When to Remove](#outliers)
6. [Decision Tree Splitting Criteria](#decision-tree)
7. [Normality Tests](#normality)
8. [Data Storage Formats — Parquet vs CSV](#formats)
9. [Pearson Correlation Coefficient](#correlation)
10. [Memory: Python List vs NumPy Array](#memory)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

print('Setup complete ✅')

---
## 1 · Missing Data Handling <a id='missing'></a>

### Key question
> Which method is **easy to implement but could introduce bias** if data is not MCAR?
> ✅ **More than one is correct** — all simple fill methods can introduce bias.

### Methods comparison

| Method | Code | Bias risk | When appropriate |
|---|---|---|---|
| `dropna()` | `df.dropna()` | ✅ if not MCAR | Data is truly random-missing & small % |
| Forward fill | `df.fillna(method='ffill')` | ✅ assumes time order | Time series with natural continuity |
| Backward fill | `df.fillna(method='bfill')` | ✅ uses future data | Time series (careful) |
| Mean imputation | `df.fillna(df.mean())` | ✅ distorts variance | Quick baseline only |
| KNN imputation | `KNNImputer()` | Lower | Mixed numeric data |
| Iterative (MICE) | `IterativeImputer()` | Lowest | Best statistical practice |

### Missing data mechanisms
- **MCAR** — Missing Completely At Random → safe to drop
- **MAR** — Missing At Random (depends on other columns) → impute
- **MNAR** — Missing Not At Random (value itself drives missingness) → model the mechanism

In [ ]:
# ── Missing data: all methods compared ─────────────────────────
from sklearn.impute import KNNImputer, SimpleImputer

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'age':    [25, np.nan, 35, np.nan, 45, 28, np.nan, 52],
    'income': [50000, 60000, np.nan, 80000, np.nan, 55000, 72000, 90000],
    'score':  [7.2, 6.8, 8.1, np.nan, 7.5, np.nan, 6.9, 8.4],
})

print('── Original data (with NaN) ──')
display(df)
print(f'Missing values:\n{df.isnull().sum()}\n')

# 1. dropna
print(f'After dropna(): {len(df.dropna())} rows remain (lost {len(df)-len(df.dropna())})')

# 2. Mean imputation — introduces bias toward mean
df_mean = df.fillna(df.mean())
print('\n── Mean imputation ── (note: all NaN → same mean value)')
display(df_mean.round(1))

# 3. Forward fill
df_ffill = df.fillna(method='ffill')
print('── Forward fill ──')
display(df_ffill)

# 4. KNN imputation — uses neighbouring rows
imputer = KNNImputer(n_neighbors=2)
df_knn = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print('── KNN imputation (n_neighbors=2) — better distribution preservation ──')
display(df_knn.round(1))

---
## 2 · Overfitting — Causes & Solutions <a id='overfitting'></a>

### Key question
> Which methods address overfitting? ✅ **More than one is correct**
> - **Increase training data** → model can't memorize, must generalize
> - **Decrease model complexity** → fewer parameters = less capacity to fit noise

$$\text{Overfitting} = \text{high variance} = \underbrace{\text{train accuracy} - \text{val accuracy}}_{\text{gap}} \text{ is large}$$

### Full toolkit

| Technique | Category |
|---|---|
| More training data / augmentation | Data |
| L1 / L2 regularization | Model |
| Dropout | Model |
| Reduce depth / width | Model |
| Early stopping | Training |
| Cross-validation | Evaluation |

In [ ]:
# ── Overfitting demo: shallow vs deep tree ─────────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=20, n_informative=5, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'{'max_depth':>12}  {'Train Acc':>10}  {'Val Acc':>10}  {'Gap':>8}  Status')
print('-' * 60)
for depth in [1, 2, 4, 6, 10, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_tr, y_tr)
    tr_acc  = accuracy_score(y_tr,  model.predict(X_tr))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    gap     = tr_acc - val_acc
    status  = '✅ OK' if gap < 0.05 else ('⚠️ Mild' if gap < 0.15 else '❌ Overfit')
    print(f'{str(depth):>12}  {tr_acc:>10.3f}  {val_acc:>10.3f}  {gap:>8.3f}  {status}')

---
## 3 · High Cardinality Encoding <a id='encoding'></a>

### Key question
> Best technique for high-cardinality categoricals without curse of dimensionality?
> ✅ **Frequency encoding** — always produces 1 column regardless of cardinality

| Technique | Columns produced | Best for |
|---|---|---|
| One-hot (`pd.get_dummies`) | = cardinality | Low cardinality (2–15) |
| Label (`pd.factorize`) | 1 — but implies order | Ordinal data |
| **Frequency encoding** ✅ | **1** | High cardinality |
| Target encoding | 1 | High cardinality + supervised |
| Hash encoding | n (fixed) | Very high cardinality |

In [ ]:
# ── Encoding comparison ─────────────────────────────────────────
df = pd.DataFrame({'city': ['NYC','LA','NYC','Chicago','LA','NYC','Miami','LA','Chicago','NYC']})

# One-hot: 1 column per city (bad for 10,000 cities)
ohe = pd.get_dummies(df['city'], prefix='city')
print(f'One-hot: {ohe.shape[1]} columns')
display(ohe.head(5))

# Label encoding: arbitrary integers (implies false order)
df['city_label'], _ = pd.factorize(df['city'])
print(f'\nLabel encoding (1 column, but NYC=0, LA=1... implies order):')
display(df[['city','city_label']].head(5))

# Frequency encoding: replace with proportion ✅
freq = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq)
print(f'\nFrequency encoding (1 column, preserves info):')
display(df[['city','city_freq']].drop_duplicates().sort_values('city_freq', ascending=False))

print(f'\nShape comparison — 10 rows, 1 city column:')
print(f'  One-hot:   {ohe.shape}  ← explodes with many cities')
print(f'  Frequency: (10, 1)  ← always 1 column')

---
## 4 · NumPy — Reproducibility & Arrays <a id='numpy'></a>

### Key question
> What does `np.random.seed(0)` + `np.random.rand(5)` do?
> ✅ **More than one is correct:**
> - Generates array of random numbers between 0 and 1
> - Ensures reproducibility of random number generation

| Function | Purpose |
|---|---|
| `np.random.seed(n)` | Fix RNG state → same sequence every run |
| `np.random.rand(n)` | n uniform floats in $[0, 1)$ |
| `np.random.randn(n)` | n standard normal floats $N(0,1)$ |
| `np.random.randint(lo, hi, n)` | n random integers in $[lo, hi)$ |

In [ ]:
# ── Reproducibility demo ────────────────────────────────────────
# Without seed — different every run
print('Without seed (different each run):')
print(' Run 1:', np.random.rand(5).round(4))
print(' Run 2:', np.random.rand(5).round(4))

# With seed — identical every run
print('\nWith seed(0) — always the same:')
np.random.seed(0)
r1 = np.random.rand(5)
np.random.seed(0)
r2 = np.random.rand(5)
print(' Run 1:', r1.round(4))
print(' Run 2:', r2.round(4))
print(' Equal:', np.array_equal(r1, r2))

# Modern recommended approach
rng = np.random.default_rng(seed=42)   # preferred in NumPy 1.17+
print('\nModern approach (default_rng):', rng.random(5).round(4))
print('Range: all values in [0, 1)?', np.all((r1 >= 0) & (r1 < 1)))

---
## 5 · Outliers — When to Keep, When to Remove <a id='outliers'></a>

### Key question
> Why might outliers be **preserved** instead of removed?
> ✅ **Outliers could represent valuable information and meaningful deviations**

| Domain | Outlier IS the signal |
|---|---|
| Fraud detection | The fraud IS the outlier |
| Medical diagnosis | Rare disease = extreme value |
| Manufacturing QC | Defective products = outliers |
| Financial risk | Black swan events define risk |

### Decision framework
```
Is it a measurement/data error?
  ├── Yes → Fix or remove
  └── No  → Is it meaningful to the task?
              ├── Yes (fraud, anomaly) → PRESERVE
              └── No (irrelevant extreme) → Cap (winsorize) or remove
```

In [ ]:
# ── Outlier detection & treatment methods ──────────────────────
rng = np.random.default_rng(0)
data = pd.Series(np.concatenate([
    rng.normal(50, 10, 95),  # normal data
    [150, 160, 5, -10, 200]  # outliers
]))

# IQR method
Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
IQR    = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = data[(data < lower) | (data > upper)]
print(f'IQR bounds: [{lower:.1f}, {upper:.1f}]')
print(f'Outliers detected: {len(outliers)} → {outliers.values.round(1)}')

# Z-score method
z_scores = (data - data.mean()) / data.std()
z_outliers = data[z_scores.abs() > 3]
print(f'\nZ-score |z|>3 outliers: {len(z_outliers)} → {z_outliers.values.round(1)}')

# Winsorizing (cap instead of remove)
data_winsorized = data.clip(lower=lower, upper=upper)
print(f'\nOriginal  max={data.max():.0f}  →  Winsorized max={data_winsorized.max():.1f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
data.plot.hist(bins=30, ax=axes[0], color='steelblue', title='Original (with outliers)')
data[~data.isin(outliers)].plot.hist(bins=30, ax=axes[1], color='crimson', title='Removed outliers')
data_winsorized.plot.hist(bins=30, ax=axes[2], color='green', title='Winsorized (capped)')
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 6 · Decision Tree Splitting Criteria <a id='decision-tree'></a>

### Key question
> Which method determines the best split? ✅ **All of the above**

| Criterion | Formula | Same goal |
|---|---|---|
| **Gini impurity** | $G = 1 - \sum_k p_k^2$ | Minimize → pure nodes |
| **Entropy** | $H = -\sum_k p_k \log_2 p_k$ | Minimize → pure nodes |
| **Information gain** | $IG = H(\text{parent}) - \sum \frac{n_i}{n}H(\text{child}_i)$ | Maximize → pure nodes |

All three drive toward the same goal: **pure child nodes** where each contains one class.

In [ ]:
# ── Gini vs Entropy: formula comparison ────────────────────────
p = np.linspace(0.001, 0.999, 300)  # p = probability of class 1

gini    = 2 * p * (1 - p)                             # binary Gini
entropy = -(p*np.log2(p) + (1-p)*np.log2(1-p))        # binary entropy

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, gini,    label='Gini impurity',  color='steelblue', lw=2)
ax.plot(p, entropy, label='Entropy (H)',    color='crimson',   lw=2, ls='--')
ax.axvline(0.5, color='gray', ls=':', lw=1, label='p=0.5 (max impurity)')
ax.set_xlabel('p (probability of class 1)')
ax.set_ylabel('Impurity')
ax.set_title('Gini vs Entropy — Both Measure Node Impurity', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

# sklearn uses both
from sklearn.tree import DecisionTreeClassifier
X, y = make_classification(n_samples=200, n_features=5, random_state=0)
for crit in ['gini', 'entropy']:
    model = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=0)
    model.fit(X, y)
    print(f'{crit:10s} → train acc = {model.score(X, y):.4f}')

---
## 7 · Normality Tests <a id='normality'></a>

### Key question
> Which test is used to confirm normality? ✅ **A normality test** (e.g. Shapiro-Wilk)

$H_0$: data comes from a normal distribution  
$H_a$: data does not come from a normal distribution

| Test | Best for | Python |
|---|---|---|
| **Shapiro-Wilk** | Small samples (n < 50) — most powerful | `scipy.stats.shapiro(x)` |
| **D'Agostino-Pearson** | Medium/large samples | `scipy.stats.normaltest(x)` |
| **Kolmogorov-Smirnov** | Large samples | `scipy.stats.kstest(x, 'norm')` |
| **Anderson-Darling** | Sensitive in tails | `scipy.stats.anderson(x)` |

In [ ]:
# ── Normality tests ─────────────────────────────────────────────
from scipy import stats

rng = np.random.default_rng(42)
normal_data = rng.normal(loc=50, scale=10, size=100)
skewed_data = rng.exponential(scale=10, size=100)

print('── Shapiro-Wilk test ──')
for name, data in [('Normal data', normal_data), ('Skewed data (exp)', skewed_data)]:
    stat, p = stats.shapiro(data)
    verdict = 'NORMAL ✅' if p > 0.05 else 'NOT normal ❌'
    print(f'  {name:25s}  W={stat:.4f}  p={p:.4f}  → {verdict}')

print('\n── D\'Agostino-Pearson test ──')
for name, data in [('Normal data', normal_data), ('Skewed data (exp)', skewed_data)]:
    stat, p = stats.normaltest(data)
    verdict = 'NORMAL ✅' if p > 0.05 else 'NOT normal ❌'
    print(f'  {name:25s}  stat={stat:.4f}  p={p:.4f}  → {verdict}')

# Q-Q plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(axes,
    [normal_data, skewed_data],
    ['Normal data — Q-Q Plot', 'Skewed data — Q-Q Plot']):
    stats.probplot(data, dist='norm', plot=ax)
    ax.set_title(title, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 8 · Data Storage Formats — Parquet vs CSV <a id='formats'></a>

### Key question
> Best format for columnar access on large datasets?
> ✅ **Parquet — columnar storage, optimized for analytics**

| Format | Storage model | Columnar | Compression | Best for |
|---|---|---|---|---|
| CSV | Row | ❌ | None | Small files, universal exchange |
| SQLite | Row (B-tree) | ❌ | Optional | OLTP, transactional queries |
| **Parquet** ✅ | **Column** | ✅ | Snappy/zstd | Analytics, big data, OLAP |

**Parquet advantages:**
- **Column pruning** — read only needed columns
- **Predicate pushdown** — filter without reading all data
- **Compression** — similar values per column compress ~5–10×

In [ ]:
# ── CSV vs Parquet: size & read comparison ─────────────────────
import os, tempfile, time

rng  = np.random.default_rng(0)
n    = 100_000
df   = pd.DataFrame({
    'id':     np.arange(n),
    'name':   [f'customer_{i}' for i in range(n)],
    'age':    rng.integers(18, 80, n),
    'salary': rng.lognormal(10, 1, n).round(2),
    'dept':   rng.choice(['Engineering','Marketing','HR','Finance'], n),
    'score':  rng.uniform(0, 100, n).round(2),
})

with tempfile.TemporaryDirectory() as tmp:
    csv_path     = os.path.join(tmp, 'data.csv')
    parquet_path = os.path.join(tmp, 'data.parquet')

    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, compression='snappy')

    csv_size     = os.path.getsize(csv_path)     / 1e6
    parquet_size = os.path.getsize(parquet_path) / 1e6
    print(f'File sizes ({n:,} rows):')
    print(f'  CSV:     {csv_size:.2f} MB')
    print(f'  Parquet: {parquet_size:.2f} MB  ({csv_size/parquet_size:.1f}× smaller)')

    # Read speed — full file
    t0 = time.perf_counter(); pd.read_csv(csv_path);     csv_time = time.perf_counter()-t0
    t0 = time.perf_counter(); pd.read_parquet(parquet_path); pq_time = time.perf_counter()-t0
    print(f'\nRead full file:')
    print(f'  CSV:     {csv_time*1000:.1f} ms')
    print(f'  Parquet: {pq_time*1000:.1f} ms')

    # Column pruning — Parquet reads only what you ask for
    t0 = time.perf_counter(); pd.read_parquet(parquet_path, columns=['salary','dept']); pq_col_time = time.perf_counter()-t0
    print(f'  Parquet (2 cols only): {pq_col_time*1000:.1f} ms  ← column pruning')

---
## 9 · Pearson Correlation Coefficient <a id='correlation'></a>

### Key question
> Pearson r = −0.8 means? ✅ **Strong negative linear relationship**

| r range | Strength | Direction |
|---|---|---|
| $\pm 0.9$ to $\pm 1.0$ | Very strong | +/− |
| **$-0.9$ to $-0.7$** ✅ | **Strong** | **Negative** |
| $\pm 0.5$ to $\pm 0.7$ | Moderate | +/− |
| $0$ to $\pm 0.5$ | Weak | +/− |
| $0$ | No linear relationship | — |

> ⚠️ **Correlation ≠ Causation** — strong r does not imply one variable causes the other.

In [ ]:
# ── Correlation demo: from -1 to +1 ────────────────────────────
rng = np.random.default_rng(42)
n   = 200
x   = rng.normal(0, 1, n)

cases = {
    'r ≈ −0.9 (very strong neg)': -0.9 * x + 0.44 * rng.normal(0,1,n),
    'r ≈ −0.8 (strong neg)':      -0.8 * x + 0.60 * rng.normal(0,1,n),
    'r ≈  0.0 (no relationship)':  0.0 * x + 1.00 * rng.normal(0,1,n),
    'r ≈ +0.8 (strong pos)':      +0.8 * x + 0.60 * rng.normal(0,1,n),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, y) in zip(axes, cases.items()):
    r, _ = stats.pearsonr(x, y)
    ax.scatter(x, y, alpha=0.5, s=15, color='steelblue')
    ax.set_title(f'{title}\nr = {r:.2f}', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Pearson Correlation — Visual Guide', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Pandas correlation matrix
df_corr = pd.DataFrame({'x': x, 'strong_neg': cases['r ≈ −0.8 (strong neg)']})
print('\nCorrelation matrix:')
display(df_corr.corr().round(3))

---
## 10 · Memory: Python List vs NumPy Array <a id='memory'></a>

### Key question
> Which uses less memory with 1M integers?
> ✅ **NumPy array — homogeneous, no Python object overhead**

| Structure | Bytes/element | 1M elements |
|---|---|---|
| Python list | ~28 bytes (full object) | ~28 MB |
| NumPy int64 | 8 bytes | **8 MB** |
| NumPy int32 | 4 bytes | **4 MB** |
| NumPy int8 | 1 byte | **1 MB** |

In [ ]:
# ── Memory: Python list vs NumPy ───────────────────────────────
import sys

n = 1_000_000
py_list  = list(range(n))
np_int64 = np.arange(n, dtype=np.int64)
np_int32 = np.arange(n, dtype=np.int32)
np_int8  = np.arange(n, dtype=np.int8)   # only works if values fit in [-128, 127]

# Python list: header + per-element pointer + per-element object
list_mem = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in range(min(n, 1000))) * (n / 1000)

print(f'Memory for {n:,} integers:')
print(f'  Python list   : ~{list_mem/1e6:.1f} MB  ({list_mem/n:.0f} bytes/element)')
print(f'  NumPy int64   :  {np_int64.nbytes/1e6:.1f} MB  ({np_int64.itemsize} bytes/element)')
print(f'  NumPy int32   :  {np_int32.nbytes/1e6:.1f} MB  ({np_int32.itemsize} bytes/element)')

print(f'\nSpeedup: NumPy is ~{list_mem/np_int64.nbytes:.0f}× more memory-efficient than Python list')

# Speed comparison: sum operation
import time
t0 = time.perf_counter(); sum(py_list);           list_time = time.perf_counter()-t0
t0 = time.perf_counter(); np_int64.sum();         np_time   = time.perf_counter()-t0

print(f'\nSum of {n:,} integers:')
print(f'  Python sum():   {list_time*1000:.1f} ms')
print(f'  NumPy .sum():   {np_time*1000:.2f} ms  ({list_time/np_time:.0f}× faster)')